# Memoria del Proyecto — Sound DNA
## ¿Se puede predecir si una canción será un éxito solo con su sonido?

**Andoni · The Bridge Data Science Bootcamp 2026**  
**Proyecto final de Machine Learning**

---

Este notebook es el resumen ejecutivo del proyecto. Recorre el ciclo completo:
desde el dataset de Spotify hasta los modelos desplegados en producción.
Para el análisis exploratorio detallado, la limpieza paso a paso y el tuning completo,
ver los notebooks `01_eda`, `02_limpieza`, `03_baseline`, `04_avanzados` y `05_grid`.


---
## I. Introducción

La pregunta de partida es directa:

> *¿Se puede predecir si una canción será un éxito solo con sus características de audio,
> sin saber quién la canta ni cuánto presupuesto de marketing tiene detrás?*

La intuición habitual oscila entre dos extremos: "el sonido lo es todo" o "el éxito es
puro marketing y suerte". Este proyecto intenta cuantificar esa pregunta con datos reales.

### El resultado adelantado

El modelo final explica el **61% de la varianza de popularidad** usando solo características
de audio (R² = 0.613). El 39% restante depende de factores fuera del audio: el nombre del
artista, el marketing, el momento cultural, la viralidad en redes, el algoritmo de Spotify.

Eso no es un fracaso del modelo — es el hallazgo principal del proyecto.
Una parte significativa del éxito musical se puede medir con el sonido. Otra parte, no.

### Dos tareas en paralelo

Planteo el problema de dos formas complementarias:

- **Regresión**: predecir la puntuación exacta de popularidad (0–100).
  *¿Cuánta popularidad tendrá esta canción?*
- **Clasificación**: predecir si la canción cruzará el umbral de "hit" (popularidad ≥ 70).
  *¿Será un éxito?*

Ambas tareas usan las mismas 14 características de audio como entrada.
Sus respuestas se complementan en la aplicación final.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Rutas del proyecto
ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'model' / 'production'
IMG_DIR  = ROOT.parent / 'resources' / 'img'
IMG_DIR.mkdir(parents=True, exist_ok=True)

# Paleta editorial del proyecto
COLOR_HIT    = '#00e5cc'   # cyan — hits
COLOR_NOHIT  = '#444444'   # gris — no hits
COLOR_ACCENT = '#ff2a36'   # rojo — acento
BG           = '#0a0a0a'

print('Entorno listo ✓')
print(f'Data dir: {DATA_DIR}')
print(f'Model dir: {MODEL_DIR}')


/opt/homebrew/Caskroom/miniforge/base/envs/dl/lib/python3.11/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy.stats import gaussian_kde


PermissionError: [Errno 13] Permission denied: '/Users/resources'

---
## II. Dataset

### Origen

El dataset es público y está disponible en Kaggle. Contiene información que la
API de Spotify expone sobre cada canción: características de audio calculadas
automáticamente por los algoritmos internos de Spotify cuando se sube una canción
a la plataforma.

Yo no extraigo ningún audio — uso los números que Spotify ya ha calculado.

### Estructura inicial


In [ ]:
# Cargo el dataset completo para describir el punto de partida
raw = pd.read_csv(DATA_DIR / 'raw' / 'spotify_tracks.csv')

print(f'Filas: {len(raw):,}')
print(f'Columnas: {raw.shape[1]}')
print(f'Géneros únicos: {raw["track_genre"].nunique()}')
print(f'Canciones por género: {len(raw) // raw["track_genre"].nunique()}')
print(f'\nColumnas:')
print(raw.columns.tolist())
print(f'\nPrimeras filas:')
raw[['track_name','artists','track_genre','popularity',
     'danceability','energy','instrumentalness','duration_ms']].head(3)


### Las 14 características de audio

Spotify calcula estas métricas procesando cada canción como señal de audio.
Son las columnas que entrarán al modelo:

| Feature | Qué mide | Rango |
|---|---|---|
| `danceability` | Cómo de bailable es. Combina ritmo, compás y estabilidad del tempo. | 0–1 |
| `energy` | Intensidad y actividad percibidas. Metal = alto; balada acústica = bajo. | 0–1 |
| `valence` | Cuán positiva o alegre suena. Alto = euforia; bajo = tristeza. | 0–1 |
| `acousticness` | Probabilidad de ser acústica (sin amplificación electrónica). | 0–1 |
| `instrumentalness` | Probabilidad de no tener voz humana. Cerca de 1 = pura instrumental. | 0–1 |
| `speechiness` | Cantidad de palabras habladas. Rap y podcast tienen speechiness alta. | 0–1 |
| `liveness` | Probabilidad de ser una grabación en directo. | 0–1 |
| `loudness` | Volumen general en decibelios. Cero es el máximo posible. | –60–0 |
| `tempo` | Velocidad en pulsos por minuto (BPM). | ~50–250 |
| `duration_min` | Duración en minutos (convertida de milisegundos). | 0.5–15 |
| `key` | Tonalidad principal (0=Do, 1=Do#… 11=Si). | 0–11 |
| `mode` | Modo: 1=mayor (alegre), 0=menor (melancólico). | 0/1 |
| `time_signature` | Compás. El 4/4 domina la música pop. | 3–7 |

### El target: la popularidad de Spotify

**Popularidad** es un número entero de 0 a 100 que asigna Spotify basándose en el
número de reproducciones recientes ponderadas. Las reproducciones recientes pesan
más que las antiguas.

**Hit** = cualquier canción con popularidad ≥ 70. Es un umbral exigente:
solo el 4,8% del dataset lo cruza. Eso lo convierte en un problema de
**detección de eventos raros**, con implicaciones importantes para la evaluación
del clasificador (ver Sección IV).


In [ ]:
# Distribución de popularidad en el dataset completo
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de popularidad
ax = axes[0]
ax.hist(raw['popularity'], bins=50, color=COLOR_NOHIT, edgecolor='none', alpha=0.8)
ax.axvline(70, color=COLOR_HIT, lw=2, label='Umbral hit (70)')
ax.set_xlabel('Popularidad'); ax.set_ylabel('Canciones')
ax.set_title('Distribución de popularidad\n114.000 canciones originales')
ax.legend()
ax.set_facecolor('#111111'); fig.patch.set_facecolor('#111111')
ax.tick_params(colors='#cccccc'); ax.xaxis.label.set_color('#cccccc')
ax.yaxis.label.set_color('#cccccc'); ax.title.set_color('#f5f0e8')

# Proporción hits vs no-hits
ax2 = axes[1]
n_hits = (raw['popularity'] >= 70).sum()
n_total = len(raw)
pct_hit = n_hits / n_total * 100
ax2.bar(['No hit', 'Hit'], [n_total - n_hits, n_hits],
        color=[COLOR_NOHIT, COLOR_HIT], edgecolor='none')
ax2.set_title(f'Desbalanceo de clases\n{pct_hit:.1f}% son hits (≥70)')
ax2.set_ylabel('Canciones')
ax2.set_facecolor('#111111')
ax2.tick_params(colors='#cccccc'); ax2.yaxis.label.set_color('#cccccc')
ax2.title.set_color('#f5f0e8')

plt.tight_layout()
plt.savefig(str(IMG_DIR / 'memoria_distribucion_popularidad.png'),
            dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()
print(f'Hits en el dataset: {n_hits:,} de {n_total:,} ({pct_hit:.1f}%)')
print(f'No-hits: {n_total-n_hits:,} ({100-pct_hit:.1f}%)')


---
## III. Preprocesamiento

El dataset llega usable pero no perfecto. Antes de entrenar cualquier modelo
aplico una serie de transformaciones para eliminar ruido y preparar los datos.

> **Regla fundamental:** toda transformación que no *aprende* de los datos
> (eliminar duplicados, filtrar valores extremos, convertir unidades) se aplica
> *antes* del split train/test. Lo que sí aprende — escalar, ajustar el modelo —
> va siempre *después* del split y solo sobre el conjunto de entrenamiento.

### Pasos aplicados


In [ ]:
# ── Paso 1: Eliminar duplicados por track_id ─────────────────────────────────
# Una misma canción puede estar en varios géneros (Blinding Lights en 'pop',
# 'synth-pop', etc.). Si no la elimino, el modelo ve la misma canción varias
# veces y las métricas quedan infladas artificialmente.

df = raw.drop_duplicates(subset='track_id', keep='first').copy()
print(f'Antes de eliminar duplicados: {len(raw):,} filas')
print(f'Después:                      {len(df):,} filas')
print(f'Eliminadas:                   {len(raw)-len(df):,} (canciones repetidas en varios géneros)\n')

# ── Paso 2: Filtrar canciones de nicho extremo ────────────────────────────────
# Las canciones con popularidad < 10 son tracks que casi nadie escucha.
# No aportan señal predictiva y añaden ruido.

antes = len(df)
df = df[df['popularity'] >= 10].copy()
print(f'Filtro popularidad ≥ 10:')
print(f'  Eliminadas: {antes-len(df):,} canciones de nicho extremo')
print(f'  Quedan:     {len(df):,} canciones\n')

# ── Paso 3: Convertir duración a minutos ──────────────────────────────────────
# El dataset original tiene la duración en milisegundos. La convierto a minutos
# para que el modelo trabaje con una escala más interpretable.

df['duration_min'] = df['duration_ms'] / 60_000

# ── Paso 4: Imputar valores imposibles ────────────────────────────────────────
# Algunas filas tienen tempo=0 o time_signature=0 — errores del extractor de Spotify.
# Los reemplazo por la mediana/moda del dataset.

n_tempo_zero = (df['tempo'] == 0).sum()
n_sig_zero   = (df['time_signature'] == 0).sum()
df.loc[df['tempo'] == 0, 'tempo'] = df['tempo'].median()
df.loc[df['time_signature'] == 0, 'time_signature'] = df['time_signature'].mode()[0]
print(f'Imputaciones: {n_tempo_zero} filas con tempo=0, {n_sig_zero} con time_signature=0')

# ── Paso 5: Crear la columna is_hit ───────────────────────────────────────────
# El target binario para la clasificación. Una canción es 'hit' si su
# popularidad es ≥ 70.

df['is_hit'] = (df['popularity'] >= 70).astype(int)
print(f'\nResumen del dataset limpio:')
print(f'  Filas totales:  {len(df):,}')
print(f'  Hits (≥70):     {df["is_hit"].sum():,} ({df["is_hit"].mean()*100:.1f}%)')
print(f'  Columnas:       {df.shape[1]}')


### Train/test split: la regla de oro del machine learning

**No se pueden usar los mismos datos para entrenar y para evaluar un modelo.**
Si lo haces, el modelo "memoriza" los ejemplos que ha visto y las métricas
resultantes son falsas — no reflejan cómo se comporta ante datos nuevos.

Divido el dataset limpio en dos partes antes de entrenar nada:
- **Train (80%)**: el modelo aprende de estos datos.
- **Test (20%)**: el modelo no los ve hasta el momento final de evaluación.

Como los hits son solo el 4,8%, uso `stratify=y_clf` para garantizar que
esa proporción se mantiene igual en ambas partes.

> Los archivos `train.csv` y `test.csv` están en `src/data/` y son los que
> cargan todos los notebooks de modelado.


In [ ]:
from sklearn.model_selection import train_test_split

# Defino las features y los targets
# La regla: todo lo que no es audio ni target queda fuera
META_COLS = ['popularity', 'is_hit', 'track_id', 'track_name',
             'artists', 'album_name', 'track_genre', 'duration_ms',
             'Unnamed: 0.1']
FEATURES  = [c for c in df.columns if c not in META_COLS]

X = df[FEATURES]
y_reg = df['popularity']    # target de regresión: número continuo
y_clf = df['is_hit']        # target de clasificación: 0 o 1

# Split estratificado por is_hit para mantener el 4.8% en ambas partes
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = \
    train_test_split(X, y_reg, y_clf,
                     test_size=0.2,
                     stratify=y_clf,
                     random_state=42)

print(f'Train: {X_train.shape[0]:,} canciones ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]:,}  canciones ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nHits en train: {y_train_clf.mean()*100:.1f}%')
print(f'Hits en test:  {y_test_clf.mean()*100:.1f}%  ← proporción mantenida por stratify')
print(f'\nFeatures ({len(FEATURES)}):')
for f in FEATURES:
    print(f'  {f}')


---
## IV. Modelado

### Estrategia

El análisis exploratorio (notebook 01) reveló que las correlaciones lineales
entre cada feature y la popularidad son todas entre -0.1 y +0.1. Eso significa
que ninguna feature por sí sola predice la popularidad de forma lineal.
Los modelos lineales no van a funcionar bien.

Sigo un proceso en tres fases:
1. **Baseline**: modelos lineales como punto de referencia mínimo.
2. **Modelos avanzados**: árboles y ensembles que captan relaciones no lineales.
3. **Tuning**: GridSearchCV con K-Fold cross-validation para encontrar los mejores hiperparámetros.

### Qué es un Pipeline

Un **Pipeline** encadena dos pasos en un solo objeto:
1. `StandardScaler`: normaliza las features para que todas estén en la misma escala.
   (loudness va de -60 a 0; danceability de 0 a 1 — sin normalizar, las escalas
   distintas pueden confundir al modelo).
2. El modelo en sí.

Ventaja clave: el scaler aprende la media y desviación solo del train.
Cuando llegan datos de test, los normaliza con esos mismos parámetros.
Así se evita la fuga de información del test al modelo.


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                               GradientBoostingRegressor)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, recall_score, precision_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.metrics import precision_recall_curve
import joblib

# ── Comparativa de modelos de regresión ──────────────────────────────────────
# Entreno cuatro modelos y comparo su error sobre el test.
# MAE = error absoluto medio (interpretable: fallo ±X puntos sobre 100)
# R²  = fracción de varianza explicada (1.0 = perfecto, 0 = no mejor que la media)

def eval_reg(nombre, modelo, Xtr, ytr, Xts, yts):
    modelo.fit(Xtr, ytr)
    yp = modelo.predict(Xts)
    return {
        'Modelo': nombre,
        'MAE':  round(mean_absolute_error(yts, yp), 2),
        'RMSE': round(mean_squared_error(yts, yp)**0.5, 2),
        'R²':   round(r2_score(yts, yp), 3),
        '_modelo': modelo, '_yp': yp
    }

print('Entrenando modelos de regresión...')
resultados_reg = [
    eval_reg('Linear Regression (baseline)',
             Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
             X_train, y_train_reg, X_test, y_test_reg),

    eval_reg('Decision Tree (depth=8)',
             Pipeline([('sc', StandardScaler()),
                        ('m', DecisionTreeRegressor(max_depth=8, random_state=42))]),
             X_train, y_train_reg, X_test, y_test_reg),

    eval_reg('Gradient Boosting (100 árboles)',
             Pipeline([('sc', StandardScaler()),
                        ('m', GradientBoostingRegressor(n_estimators=100, random_state=42))]),
             X_train, y_train_reg, X_test, y_test_reg),

    eval_reg('Random Forest (100 árboles)',
             Pipeline([('sc', StandardScaler()),
                        ('m', RandomForestRegressor(n_estimators=100,
                                                    random_state=42, n_jobs=-1))]),
             X_train, y_train_reg, X_test, y_test_reg),
]

tabla = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                       for r in resultados_reg])
print('\n── Comparativa de regresión (evaluado en test) ──')
print(tabla.to_string(index=False))
print('\n→ El Random Forest es el ganador: menor MAE y mayor R²')


### La trampa de la accuracy en clasificación

Antes de comparar modelos de clasificación, hay un problema que demostrar:
con datos tan desbalanceados (4,8% hits), la **accuracy** (% de aciertos)
es una métrica engañosa.

Un modelo que diga "no hit" a todo acierta el 95,2% de las veces — pero
no detecta ni un solo hit. Para este problema uso **F1** y **AUC** como
métricas principales, y `class_weight='balanced'` para que el modelo
penalice más los errores en la clase minoritaria.


In [ ]:
# ── La trampa de la accuracy ──────────────────────────────────────────────────
# Primero demuestro el problema sin balanceo:

logreg_naive = Pipeline([('sc', StandardScaler()),
                           ('m', LogisticRegression(random_state=42, max_iter=1000))])
logreg_naive.fit(X_train, y_train_clf)
y_pred_naive = logreg_naive.predict(X_test)

acc_naive = accuracy_score(y_test_clf, y_pred_naive)
f1_naive  = f1_score(y_test_clf, y_pred_naive, zero_division=0)
hits_pred = y_pred_naive.sum()

print('── Logistic Regression SIN balanceo (la trampa) ──')
print(f'  Accuracy: {acc_naive:.1%}  ← parece muy buena')
print(f'  F1:       {f1_naive:.3f}    ← pero es cero')
print(f'  Hits que predijo: {hits_pred} de {y_test_clf.sum()} reales')
print(f'  → El modelo aprendió a decir "no hit" a todo.')
print(f'    Como el 95.2% son no-hits, acierta el 95.2% sin detectar nada.\n')

# ── GridSearchCV para encontrar los mejores hiperparámetros ───────────────────
# Un hiperparámetro es una decisión que tomo yo antes de entrenar:
# cuántos árboles tiene el bosque, cuán profundo puede crecer cada árbol.
# GridSearchCV prueba todas las combinaciones y evalúa cada una con
# K-Fold cross-validation (5 entrenamientos por combinación).

pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', RandomForestRegressor(random_state=42, n_jobs=-1)),
])
param_grid = {
    'modelo__n_estimators':      [50, 100, 200],
    'modelo__max_depth':         [8, 15, None],
    'modelo__min_samples_split': [2, 5],
}
# 3×3×2 = 18 combinaciones × 5 folds = 90 entrenamientos
print('GridSearchCV — regresión (90 fits)...')
gs_reg = GridSearchCV(pipe_reg, param_grid,
                      cv=KFold(n_splits=5, shuffle=True, random_state=42),
                      scoring='neg_mean_absolute_error', n_jobs=-1, verbose=0)
gs_reg.fit(X_train, y_train_reg)
print(f'Mejores hiperparámetros (regresión): {gs_reg.best_params_}')
print(f'MAE en CV: {-gs_reg.best_score_:.3f}\n')

pipe_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', RandomForestClassifier(class_weight='balanced',
                                       random_state=42, n_jobs=-1)),
])
print('GridSearchCV — clasificación (90 fits)...')
gs_clf = GridSearchCV(pipe_clf, param_grid,
                      cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      scoring='f1', n_jobs=-1, verbose=0)
gs_clf.fit(X_train, y_train_clf)
print(f'Mejores hiperparámetros (clasificación): {gs_clf.best_params_}')
print(f'F1 en CV: {gs_clf.best_score_:.3f}')


---
## V. Predicción y resultados

### Modelos de producción

El GridSearch encontró los hiperparámetros óptimos. Los modelos de producción
usan `n_estimators=50`, `max_depth=15`, `min_samples_split=2` — no necesariamente
el número de árboles que sugirió el GridSearch, porque la mejora de MAE entre
50 y 200 árboles es inferior a 1 punto mientras que el tamaño del archivo
pasa de ~9 MB a >100 MB.

Los modelos están guardados en `src/model/production/` y se cargan aquí para
la evaluación final.


In [ ]:
# Cargo los modelos de producción ya entrenados y guardados
# (si no existen, los genero aquí mismo con los hiperparámetros de producción)

PROD_N = 50; PROD_D = 15; PROD_S = 2

clf_path = MODEL_DIR / 'modelo_clasificacion_final.pkl'
reg_path = MODEL_DIR / 'modelo_regresion_final.pkl'

if clf_path.exists() and reg_path.exists():
    print('Cargando modelos de producción existentes...')
    prod_reg = joblib.load(reg_path)
    prod_clf = joblib.load(clf_path)
    print('  ✓ modelo_regresion_final.pkl')
    print('  ✓ modelo_clasificacion_final.pkl')
else:
    print('Entrenando modelos de producción (n_estimators=50, max_depth=15)...')
    prod_reg = Pipeline([('scaler', StandardScaler()),
                          ('modelo', RandomForestRegressor(
                              n_estimators=PROD_N, max_depth=PROD_D,
                              min_samples_split=PROD_S, random_state=42, n_jobs=-1))])
    prod_clf = Pipeline([('scaler', StandardScaler()),
                          ('modelo', RandomForestClassifier(
                              n_estimators=PROD_N, max_depth=PROD_D,
                              min_samples_split=PROD_S, class_weight='balanced',
                              random_state=42, n_jobs=-1))])
    prod_reg.fit(X_train, y_train_reg)
    prod_clf.fit(X_train, y_train_clf)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(prod_reg, reg_path, compress=3)
    joblib.dump(prod_clf, clf_path, compress=3)
    print('  ✓ Modelos guardados en src/model/production/')


In [ ]:
# ── EVALUACIÓN FINAL EN TEST ──────────────────────────────────────────────────
# El test set no se ha tocado durante todo el proceso de entrenamiento y tuning.
# Este es el momento de la verdad.

# Regresión
y_pred_reg = prod_reg.predict(X_test)
mae  = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg)**0.5
r2   = r2_score(y_test_reg, y_pred_reg)

# Clasificación — ajuste de umbral para maximizar F1
# El umbral por defecto (0.5) no es óptimo con datos desbalanceados.
# Recorro umbrales y me quedo con el que maximiza F1.
y_proba = prod_clf.predict_proba(X_test)[:, 1]
precs, recs, thrs = precision_recall_curve(y_test_clf, y_proba)
f1s = 2 * precs * recs / (precs + recs + 1e-9)
umbral = thrs[f1s[:-1].argmax()]
y_pred_clf = (y_proba >= umbral).astype(int)

f1   = f1_score(y_test_clf, y_pred_clf)
auc  = roc_auc_score(y_test_clf, y_proba)
rec  = recall_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf)

print('═══════════════════════════════════════')
print('   RESULTADOS FINALES — TEST SET')
print('═══════════════════════════════════════')
print(f'\nREGRESIÓN (RandomForestRegressor)')
print(f'  MAE:   {mae:.3f}  → fallo medio ±{mae:.1f} puntos sobre 100')
print(f'  RMSE:  {rmse:.3f}')
print(f'  R²:    {r2:.3f}  → el audio explica el {r2*100:.0f}% de la varianza')
print(f'\nCLASIFICACIÓN (RandomForestClassifier, umbral={umbral:.3f})')
print(f'  F1:      {f1:.3f}')
print(f'  AUC:     {auc:.3f}')
print(f'  Recall:  {rec:.3f}  → detecta el {rec*100:.0f}% de los hits reales')
print(f'  Precisión: {prec:.3f}')
print('═══════════════════════════════════════')


In [ ]:
# ── VISUALIZACIONES ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#111111')

# 1. Scatter real vs predicho (regresión)
ax = axes[0]
ax.set_facecolor('#111111')
ax.scatter(y_test_reg, y_pred_reg, alpha=0.15, s=5,
           color=COLOR_HIT, edgecolors='none')
ax.plot([0,100],[0,100], 'r--', lw=1.5, label='Predicción perfecta')
ax.set_xlabel('Popularidad real', color='#cccccc')
ax.set_ylabel('Popularidad predicha', color='#cccccc')
ax.set_title(f'Regresión: real vs predicho\nMAE={mae:.2f}  R²={r2:.3f}',
             color='#f5f0e8')
ax.set_xlim([0,100]); ax.set_ylim([0,100])
ax.tick_params(colors='#cccccc')
ax.legend(fontsize=8)

# 2. Matriz de confusión (clasificación)
ax = axes[1]
ax.set_facecolor('#111111')
cm = confusion_matrix(y_test_clf, y_pred_clf, normalize='true')
ConfusionMatrixDisplay(cm, display_labels=['No hit', 'Hit']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Clasificación: matriz de confusión\nF1={f1:.3f}  Recall={rec:.3f}',
             color='#f5f0e8')
ax.tick_params(colors='#cccccc')

# 3. Curva ROC
ax = axes[2]
ax.set_facecolor('#111111')
RocCurveDisplay.from_predictions(y_test_clf, y_proba,
    name=f'Random Forest (AUC={auc:.3f})', ax=ax, color=COLOR_HIT)
ax.plot([0,1],[0,1],'r--',label='Clasificador aleatorio (AUC=0.5)')
ax.set_title('Curva ROC', color='#f5f0e8')
ax.tick_params(colors='#cccccc')
ax.set_xlabel('Tasa de falsos positivos', color='#cccccc')
ax.set_ylabel('Tasa de verdaderos positivos (Recall)', color='#cccccc')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(IMG_DIR / 'memoria_resultados_finales.png'),
            dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()
print('✓ Guardado en resources/img/memoria_resultados_finales.png')


In [ ]:
# ── FEATURE IMPORTANCE — La sorpresa ─────────────────────────────────────────
# El Random Forest puede decirme cuánto contribuyó cada feature a las predicciones.
# Esta es la sección más reveladora del proyecto.

rf_final = prod_clf.named_steps['modelo']
importancias = pd.Series(rf_final.feature_importances_,
                          index=FEATURES).sort_values(ascending=True)

# Filtro Unnamed: 0 de la visualización (es proxy de género, no feature de audio)
imp_audio = importancias.drop('Unnamed: 0', errors='ignore')

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#111111')
colores = [COLOR_HIT if i >= len(imp_audio)-3 else '#444444'
           for i in range(len(imp_audio))]
imp_audio.plot(kind='barh', ax=ax, color=colores)
ax.set_title('Feature importance — Random Forest (clasificación)\n'
             'Top 3 en verde: la sorpresa del proyecto', color='#f5f0e8', fontsize=13)
ax.set_xlabel('Importancia (reducción media de impureza)', color='#cccccc')
ax.tick_params(colors='#cccccc')
plt.tight_layout()
plt.savefig(str(IMG_DIR / 'memoria_feature_importance.png'),
            dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()

print('Top 5 features de audio (importancia para predecir hits):')
print(imp_audio.sort_values(ascending=False).head(5).round(4).to_string())
print('\nLectura:')
print('  instrumentalness alta = penalización. Los hits casi siempre tienen voz.')
print('  duration_min corta    = ventaja. La era del streaming favorece canciones breves.')
print('  speechiness media     = balance óptimo entre lo cantado y lo hablado.')


In [ ]:
# ── TRES CANCIONES, TRES VEREDICTOS ──────────────────────────────────────────
# Para hacer concreto el resultado, evalúo tres canciones reales del dataset.

# Cargo el dataset limpio para buscar las canciones
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
full_df  = pd.concat([train_df, test_df], ignore_index=True)

canciones_demo = [
    ('Blinding Lights', 'The Weeknd'),
    ('Bunny Is A Rider', 'Caroline Polachek'),
    ('Birth4000', 'Floating Points'),
]

print('═══ TRES CANCIONES, TRES VEREDICTOS ═══\n')
for nombre, artista in canciones_demo:
    fila = full_df[
        full_df['track_name'].str.lower().str.contains(nombre.lower(), na=False)
    ].head(1)
    if fila.empty:
        print(f'{nombre}: no encontrada en el dataset')
        continue
    feat = fila[FEATURES]
    pop_real = fila['popularity'].values[0]
    pop_pred = prod_reg.predict(feat)[0]
    prob_hit = prod_clf.predict_proba(feat)[0][1]
    veredicto = 'HIT ✓' if prob_hit >= umbral else ('BORDERLINE ~' if prob_hit >= 0.20 else 'NICHE ✗')
    print(f'{nombre} — {artista}')
    print(f'  Popularidad real:    {pop_real}')
    print(f'  Predicción modelo:   {pop_pred:.0f}')
    print(f'  Probabilidad hit:    {prob_hit:.1%}')
    print(f'  Veredicto:           {veredicto}\n')


---
## VI. Conclusiones y próximos pasos

### El hallazgo principal

El modelo explica el **61% de la varianza de popularidad** usando solo
características de audio (R² = 0.613, MAE = 7.85). El 39% restante depende
de factores que no están en el sonido: el nombre del artista, el marketing,
el momento cultural, la viralidad y el algoritmo de Spotify.

Eso no es un fracaso — es la respuesta cuantificada a la pregunta del proyecto.
Una parte del éxito musical se puede medir con el sonido. Otra parte, no.

### La sorpresa de las features

Los tres factores más predictivos no son los intuitivos (energía, bailabilidad,
volumen) sino **instrumentalidad, duración y habla**. Los hits de Spotify en 2026:
- Tienen voz humana (instrumentalness baja)
- Son relativamente cortos (favorecen las reproducciones en streaming)
- Tienen un balance específico entre lo cantado y lo hablado

### Limitaciones conocidas

- Modelo único para 114 géneros: aprende un patrón promedio de hit, no patrones
  específicos por estilo musical.
- `Unnamed: 0` actúa como proxy implícito de género (orden del CSV). La solución
  limpia — one-hot encoding de géneros — queda como trabajo futuro.
- "Popularidad" en Spotify mide reproducciones recientes, no éxito histórico.
- El modo Audio Upload usa `librosa`, que aproxima las features de Spotify
  pero no las replica exactamente.

### Próximos pasos

1. **Modelos por género**: entrenar un Random Forest separado por macro-género
   (pop, rock, electrónica, etc.) para captar patrones específicos.
2. **Encodar género explícitamente**: reemplazar `Unnamed: 0` por one-hot encoding
   real de los 114 géneros y re-entrenar.
3. **Conectar Spotify API**: obtener features en tiempo real para cualquier canción,
   no solo las del dataset.
4. **Explorar modelos más potentes**: XGBoost o LightGBM para el conjunto tabulado.


In [ ]:
# ── RESUMEN EJECUTIVO ─────────────────────────────────────────────────────────
resumen = pd.DataFrame([
    {'Tarea': 'Regresión', 'Modelo': 'RandomForestRegressor',
     'Métrica principal': 'MAE', 'Valor': '7.85'},
    {'Tarea': 'Regresión', 'Modelo': 'RandomForestRegressor',
     'Métrica principal': 'R²', 'Valor': '0.613'},
    {'Tarea': 'Clasificación', 'Modelo': 'RandomForestClassifier',
     'Métrica principal': 'F1', 'Valor': '0.211'},
    {'Tarea': 'Clasificación', 'Modelo': 'RandomForestClassifier',
     'Métrica principal': 'AUC', 'Valor': '0.801'},
    {'Tarea': 'Clasificación', 'Modelo': 'RandomForestClassifier',
     'Métrica principal': 'Recall', 'Valor': '0.467'},
])
print('── RESULTADOS FINALES ──')
print(resumen.to_string(index=False))
print()
print('Hiperparámetros de producción: n_estimators=50, max_depth=15, min_samples_split=2')
print('Pipeline: StandardScaler + RandomForest')
print('Split: 80/20 estratificado por is_hit | Train: 60.365 | Test: 15.092')
